In [1]:
import os
import json
from pathlib import Path
from typing import Optional, List
from email import policy
from email.parser import BytesParser

import pandas as pd
import extract_msg
from pydantic import BaseModel, Field

from google import genai

In [4]:
# CONFIG

MAIL_FOLDER = r"C:\Users\LotusBlue\Downloads\invoice_msg_emails"
OUTPUT_CSV = r"C:\Users\LotusBlue\Downloads\invoice_msg_emails\extracted_invoices.csv"

# Recommended model:
# - gemini-3-flash-preview is shown in Google's text generation docs
# - you can switch to another current Gemini model if you prefer
MODEL_NAME = "gemini-3-flash-preview"

In [8]:
# GEMINI CLIENT
from dotenv import load_dotenv
load_dotenv()

client = genai.Client()


In [10]:
# STRUCTURED OUTPUT SCHEMA

class InvoiceData(BaseModel):
    source_file: Optional[str] = Field(default=None, description="Source email filename")
    sender: Optional[str] = Field(default=None, description="Email sender")
    subject: Optional[str] = Field(default=None, description="Email subject")

    invoice_number: Optional[str] = None
    invoice_date: Optional[str] = None
    vendor_name: Optional[str] = None

    subtotal: Optional[float] = None
    gst_percent: Optional[float] = None
    gst_amount: Optional[float] = None
    total_invoice_amount: Optional[float] = None
    payment_due_date: Optional[str] = None

In [12]:
# HELPERS: READ EMAIL FILES

def read_simple_email_file(file_path: str):
    """
    Reads RFC822/plain-text email files.
    Works for simple text emails saved with .msg/.eml extension.
    """
    with open(file_path, "rb") as f:
        msg = BytesParser(policy=policy.default).parse(f)

    sender = msg.get("From", "")
    subject = msg.get("Subject", "")

    body = ""
    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()
            disposition = str(part.get("Content-Disposition", ""))
            if content_type == "text/plain" and "attachment" not in disposition.lower():
                try:
                    body = part.get_content()
                    break
                except Exception:
                    payload = part.get_payload(decode=True)
                    if payload:
                        body = payload.decode(errors="ignore")
                        break
    else:
        try:
            body = msg.get_content()
        except Exception:
            payload = msg.get_payload(decode=True)
            if payload:
                body = payload.decode(errors="ignore")

    return sender, subject, body


def read_outlook_msg_file(file_path: str):
    """
    Reads native Outlook .msg files using extract-msg.
    """
    msg = extract_msg.Message(file_path)
    sender = msg.sender or ""
    subject = msg.subject or ""
    body = msg.body or ""
    return sender, subject, body


def read_email_any_format(file_path: str):
    """
    Try reading as plain email first.
    If that fails, try Outlook .msg reader.
    """
    try:
        sender, subject, body = read_simple_email_file(file_path)
        # if parsing succeeds but body is empty, still try Outlook format
        if (not body or not subject) and file_path.lower().endswith(".msg"):
            try:
                sender2, subject2, body2 = read_outlook_msg_file(file_path)
                if body2:
                    return sender2, subject2, body2
            except Exception:
                pass
        return sender, subject, body
    except Exception:
        # fallback for native Outlook .msg
        sender, subject, body = read_outlook_msg_file(file_path)
        return sender, subject, body



In [14]:
# HELPER: GEMINI EXTRACTION
# =========================================================
def extract_invoice_with_gemini(source_file: str, sender: str, subject: str, body: str) -> InvoiceData:
    """
    Sends email content to Gemini and requests only invoice-related fields.
    """

    prompt = f"""
            You are an invoice extraction system.
            
            Task:
            Extract ONLY invoice-related data from this email.
            If a field is missing, return null.
            Do not infer values unless they are clearly present.
            Ignore greetings, signatures, and non-invoice text.
            
            Return structured data for these fields only:
            - invoice_number
            - invoice_date
            - vendor_name
            - subtotal
            - gst_percent
            - gst_amount
            - total_invoice_amount
            - payment_due_date
            
            Email metadata:
            source_file: {source_file}
            sender: {sender}
            subject: {subject}
            
            Email body:
            {body}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": InvoiceData,
            "temperature": 0
        }
    )

    # Depending on SDK behavior, response.parsed may already exist.
    # Fallback to JSON load if needed.
    if hasattr(response, "parsed") and response.parsed is not None:
        parsed = response.parsed
        if isinstance(parsed, InvoiceData):
            result = parsed
        else:
            result = InvoiceData(**parsed)
    else:
        # Fallback: parse raw text JSON
        raw_text = response.text
        data = json.loads(raw_text)
        result = InvoiceData(**data)

    # populate email metadata
    result.source_file = source_file
    result.sender = sender
    result.subject = subject

    return result


In [16]:
# MAIN LOOP

all_results: List[dict] = []

mail_folder = Path(MAIL_FOLDER)

if not mail_folder.exists():
    raise FileNotFoundError(f"Folder not found: {MAIL_FOLDER}")

email_files = [f for f in mail_folder.iterdir() if f.is_file() and f.suffix.lower() in [".msg", ".eml", ".txt"]]

print(f"Found {len(email_files)} mail files")

for file_path in email_files:
    print(f"Processing: {file_path.name}")
    try:
        sender, subject, body = read_email_any_format(str(file_path))

        if not body or not body.strip():
            print(f"  Skipped - empty body: {file_path.name}")
            continue

        invoice_data = extract_invoice_with_gemini(
            source_file=file_path.name,
            sender=sender,
            subject=subject,
            body=body
        )

        all_results.append(invoice_data.model_dump())
        print(f"  Done: {file_path.name}")

    except Exception as e:
        print(f"  Error in {file_path.name}: {e}")
        all_results.append({
            "source_file": file_path.name,
            "sender": None,
            "subject": None,
            "invoice_number": None,
            "invoice_date": None,
            "vendor_name": None,
            "subtotal": None,
            "gst_percent": None,
            "gst_amount": None,
            "total_invoice_amount": None,
            "payment_due_date": None,
            "error": str(e)
        })

Found 4 mail files
Processing: invoice_email_1.msg
  Done: invoice_email_1.msg
Processing: invoice_email_2.msg
  Error in invoice_email_2.msg: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Processing: invoice_email_3.msg
  Error in invoice_email_3.msg: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Processing: invoice_email_4.msg
  Done: invoice_email_4.msg


In [18]:
# SAVE TO CSV

df = pd.DataFrame(all_results)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print("\nExtraction complete")
print(f"CSV saved to: {OUTPUT_CSV}")
print(df)


Extraction complete
CSV saved to: C:\Users\LotusBlue\Downloads\invoice_msg_emails\extracted_invoices.csv
           source_file                        sender  \
0  invoice_email_1.msg  billing@alphaelectronics.com   
1  invoice_email_2.msg                          None   
2  invoice_email_3.msg                          None   
3  invoice_email_4.msg       finance@smartoffice.com   

                                 subject invoice_number invoice_date  \
0  Invoice #INV-1001 – Alpha Electronics       INV-1001  05-Mar-2026   
1                                   None           None         None   
2                                   None           None         None   
3       Invoice #INV-1004 – Smart Office       INV-1004  08-Mar-2026   

                 vendor_name  subtotal  gst_percent  gst_amount  \
0  Alpha Electronics Pvt Ltd   15750.0         18.0      2835.0   
1                       None       NaN          NaN         NaN   
2                       None       NaN          NaN